# 🏥 Medical RAG Chatbot — MultiModel Edition
### LangChain · LangGraph · **Gemini + OpenAI + Groq** · Wikipedia · FAISS + BM25

---

| Component | Technology |
|-----------|------------|
| 🤖 LLMs | `Gemini 2.5 Flash` · `GPT-4o-mini` · `Llama-3.3-70b (Groq)` |
| 🧠 Embeddings | `sentence-transformers/all-MiniLM-L6-v2` (local) |
| 📚 Vector Stores | FAISS (semantic) · BM25 (keyword) · Hybrid Ensemble |
| 🌐 Knowledge Base | Wikipedia via `wikipedia-api` |
| 🔗 Orchestration | LangGraph ReAct Agent |
| 🔧 Tools | Hybrid RAG · FAISS RAG · BM25 RAG · Live Wikipedia · Symptom Checker · Drug Info |

> **What's New in MultiModel Edition:**  
> ✅ Switch between **3 LLMs** at runtime — Gemini, GPT-4o-mini, Groq Llama  
> ✅ Every answer shows **which RAG tool(s) were used** with descriptions  
> ✅ **Hybrid RAG**: FAISS (semantic) + BM25 (keyword) via Reciprocal Rank Fusion  
> ✅ New **Drug Information** tool with dosage, side effects, interactions  
> ✅ Urgency levels in Symptom Checker  
> ✅ Model comparison feature — same question across all LLMs

## 📦 Step 1 — Install Dependencies

In [100]:
!pip install -q \
    langchain langchain-core langchain-community \
    langchain-google-genai langchain-openai langchain-groq \
    google-generativeai langchain-huggingface \
    langgraph sentence-transformers faiss-cpu rank-bm25 \
    wikipedia-api python-dotenv tiktoken ipywidgets

print('✅ All packages installed!')

✅ All packages installed!


In [101]:
!pip install -q langchain-experimental

print('✅ langchain-experimental installed!')

✅ langchain-experimental installed!


## 🔑 Step 2 — API Keys

| Provider | Get Key | Free Tier |
|----------|---------|----------|
| **Google Gemini** | https://aistudio.google.com/app/apikey | 1500 req/day |
| **OpenAI** | https://platform.openai.com/api-keys | Pay-per-use |
| **Groq** | https://console.groq.com | 14400 tokens/min free |

> ⚠️ At least **one** key required. Press Enter to skip others.

In [102]:
import os
from getpass import getpass

print('=== API Key Setup ===')
print('Press Enter to skip a provider.\n')

gemini_key = getpass('Google Gemini API key (recommended): ').strip()
openai_key = getpass('OpenAI API key (optional): ').strip()
groq_key   = getpass('Groq API key (optional): ').strip()

if gemini_key: os.environ['GOOGLE_API_KEY'] = gemini_key
if openai_key: os.environ['OPENAI_API_KEY'] = openai_key
if groq_key:   os.environ['GROQ_API_KEY']   = groq_key

available_models = []
if gemini_key: available_models.append('gemini')
if openai_key: available_models.append('openai')
if groq_key:   available_models.append('groq')

if not available_models:
    raise ValueError('At least one API key is required!')

print(f'\n Available models: {available_models}')

=== API Key Setup ===
Press Enter to skip a provider.

Google Gemini API key (recommended): ··········
OpenAI API key (optional): ··········
Groq API key (optional): ··········

 Available models: ['gemini', 'groq']


## 📥 Step 3 — Imports

In [103]:
import os
from typing import Annotated, List, Optional
from typing_extensions import TypedDict
from IPython.display import display, Image

from langchain_google_genai import ChatGoogleGenerativeAI
try:
    from langchain_openai import ChatOpenAI
except ImportError:
    ChatOpenAI = None
try:
    from langchain_groq import ChatGroq
except ImportError:
    ChatGroq = None

from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_community.retrievers import BM25Retriever
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.tools import create_retriever_tool, tool
from langchain_core.documents import Document
from langchain_core.messages import HumanMessage, AIMessage, SystemMessage
from langgraph.graph import StateGraph, END
from langgraph.graph.message import add_messages
from langgraph.prebuilt import ToolNode
import os
from typing import Annotated, List
from typing_extensions import TypedDict
from IPython.display import display, Image

# Hugging Face
from langchain_huggingface import HuggingFaceEmbeddings, ChatHuggingFace, HuggingFaceEndpoint

# LangChain core
from langchain_community.vectorstores import FAISS
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.tools import create_retriever_tool, tool
from langchain_core.documents import Document
from langchain_core.messages import HumanMessage, AIMessage, SystemMessage

# LangGraph
from langgraph.graph import StateGraph, END
from langgraph.graph.message import add_messages
from langgraph.prebuilt import ToolNode


print(' All imports successful!')

 All imports successful!


In [104]:
!pip show langchain langchain-community

Name: langchain
Version: 1.2.15
Summary: Building applications with LLMs through composability
Home-page: https://docs.langchain.com/
Author: 
Author-email: 
License: MIT
Location: /usr/local/lib/python3.12/dist-packages
Requires: langchain-core, langgraph, pydantic
Required-by: 
---
Name: langchain-community
Version: 0.4.2
Summary: Community contributed LangChain integrations.
Home-page: https://docs.langchain.com/
Author: 
Author-email: 
License: MIT
Location: /usr/local/lib/python3.12/dist-packages
Requires: aiohttp, httpx-sse, langchain-classic, langchain-core, langsmith, numpy, pydantic-settings, pyyaml, requests, sqlalchemy, tenacity
Required-by: langchain-experimental


## 📚 Step 4 — Load Wikipedia Medical Knowledge Base

In [105]:
import wikipediaapi
from time import sleep

wiki = wikipediaapi.Wikipedia(language='en', user_agent='MediBot/2.0')

MEDICAL_TOPICS = [
    # Diseases
    'Diabetes mellitus', 'Hypertension', 'COVID-19', 'Cancer', 'Asthma',
    'Tuberculosis', 'Pneumonia', 'Influenza', 'Dengue fever', 'Malaria',
    'Heart disease', 'Stroke', 'Kidney failure', 'Liver disease', 'Obesity',
    'Anemia', 'Arthritis', 'Migraine', 'Epilepsy',
    "Parkinson's disease", "Alzheimer's disease",
    # Anatomy
    'Human body', 'Brain', 'Heart', 'Lung', 'Kidney', 'Liver',
    'Digestive system', 'Nervous system', 'Immune system',
    'Respiratory system', 'Circulatory system', 'Endocrine system',
    # Symptoms
    'Fever', 'Cough', 'Chest pain', 'Headache', 'Fatigue',
    'Shortness of breath', 'Abdominal pain', 'Diarrhea', 'Weight loss',
    # Medicines & Treatments
    'Antibiotic', 'Vaccination', 'Insulin', 'Chemotherapy',
    'Paracetamol', 'Ibuprofen', 'Pain management', 'Metformin',
    'Aspirin', 'Amoxicillin',
    # Health
    'Nutrition', 'Vitamin', 'Mental health', 'Exercise', 'Public health'
]

def fetch_medical_page(title):
    try:
        page = wiki.page(title)
        if page.exists():
            return Document(
                page_content=page.text[:5000],
                metadata={'title': page.title, 'source': page.fullurl}
            )
        return None
    except Exception as e:
        print(f'  Error loading {title}: {e}')
        return None

all_docs = []
print('Loading Medical Knowledge Base...\n')
for topic in MEDICAL_TOPICS:
    doc = fetch_medical_page(topic)
    if doc:
        all_docs.append(doc)
        print(f'  {topic}')
    else:
        print(f'  Skipped: {topic}')
    sleep(0.3)

print(f'\n📦 Total Documents Loaded: {len(all_docs)}')

Loading Medical Knowledge Base...

  Diabetes mellitus
  Hypertension
  COVID-19
  Cancer
  Asthma
  Tuberculosis
  Pneumonia
  Influenza
  Dengue fever
  Malaria
  Heart disease
  Stroke
  Kidney failure
  Liver disease
  Obesity
  Anemia
  Arthritis
  Migraine
  Epilepsy
  Parkinson's disease
  Alzheimer's disease
  Human body
  Brain
  Heart
  Lung
  Kidney
  Liver
  Digestive system
  Nervous system
  Immune system
  Respiratory system
  Circulatory system
  Endocrine system
  Fever
  Cough
  Chest pain
  Headache
  Fatigue
  Shortness of breath
  Abdominal pain
  Diarrhea
  Weight loss
  Antibiotic
  Vaccination
  Insulin
  Chemotherapy
  Paracetamol
  Ibuprofen
  Pain management
  Metformin
  Aspirin
  Amoxicillin
  Nutrition
  Vitamin
  Mental health
  Exercise
  Public health

📦 Total Documents Loaded: 57


## 🧠 Step 4.5 — Build Hybrid RAG (FAISS + BM25)

| Retriever | Method | Best For |
|-----------|--------|----------|
| **FAISS** | Dense vector embeddings | Conceptual, paraphrased questions |
| **BM25** | TF-IDF keyword scoring | Exact medical terms & drug names |
| **Ensemble** | Reciprocal Rank Fusion | Best of both — used as primary |

In [106]:
if not all_docs:
    raise ValueError(' No documents loaded.')

splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=80)
chunks = splitter.split_documents(all_docs)
print(f'  {len(all_docs)} articles → {len(chunks)} chunks')

# FAISS semantic retriever
print('\n Loading embedding model...')
embeddings = HuggingFaceEmbeddings(
    model_name='sentence-transformers/all-MiniLM-L6-v2',
    model_kwargs={'device': 'cpu'},
    encode_kwargs={'normalize_embeddings': True}
)
print('Building FAISS index...')
vectorstore = FAISS.from_documents(documents=chunks, embedding=embeddings)
faiss_retriever = vectorstore.as_retriever(search_kwargs={'k': 4})
print(f'FAISS ready — {vectorstore.index.ntotal} vectors')

# BM25 keyword retriever
print('Building BM25 keyword index...')
bm25_retriever = BM25Retriever.from_documents(chunks)
bm25_retriever.k = 4
print('BM25 ready')

# Hybrid ensemble (60% semantic, 40% keyword)
ensemble_retriever = CustomEnsembleRetriever(
    retrievers=[faiss_retriever, bm25_retriever],
    weights=[0.6, 0.4]
)
print('\n Hybrid RAG (FAISS 60% + BM25 40%) ready!')

  57 articles → 888 chunks

 Loading embedding model...


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Building FAISS index...
FAISS ready — 888 vectors
Building BM25 keyword index...
BM25 ready

 Hybrid RAG (FAISS 60% + BM25 40%) ready!


In [107]:
from collections import defaultdict
from langchain_core.documents import Document

def reciprocal_rank_fusion(results: list[list[Document]], k=60):
    """Perform Reciprocal Rank Fusion on a list of lists of documents."""
    fused_scores = defaultdict(float)
    # Collect all unique documents
    unique_docs = {}
    for r_idx, result_list in enumerate(results):
        for rank, doc in enumerate(result_list):
            # Use a unique identifier for the document, e.g., content + source
            doc_id = (doc.page_content, doc.metadata.get('source'))
            if doc_id not in unique_docs:
                unique_docs[doc_id] = doc
            fused_scores[doc_id] += 1 / (k + rank)

    # Sort documents by fused score
    sorted_doc_ids = sorted(fused_scores.keys(), key=lambda x: fused_scores[x], reverse=True)
    fused_docs = [unique_docs[doc_id] for doc_id in sorted_doc_ids]
    return fused_docs


class CustomEnsembleRetriever:
    """Custom Ensemble Retriever using Reciprocal Rank Fusion."""
    def __init__(self, retrievers: list, weights: list = None, k: int = 60):
        if weights and len(weights) != len(retrievers):
            raise ValueError("Number of weights must match number of retrievers.")
        self.retrievers = retrievers
        self.weights = weights
        self.k = k

    def get_relevant_documents(self, query: str) -> list[Document]:
        results = []
        for retriever in self.retrievers:
            results.append(retriever.get_relevant_documents(query))

        # If weights are provided, we'd need a more complex fusion logic
        # For now, RRF is applied equally to all retriever results.
        fused_documents = reciprocal_rank_fusion(results, k=self.k)
        return fused_documents

    async def aget_relevant_documents(self, query: str) -> list[Document]:
        # This needs to be implemented if asynchronous calls are expected
        raise NotImplementedError("Asynchronous retrieval not yet implemented for CustomEnsembleRetriever")

print('Custom Reciprocal Rank Fusion and Ensemble Retriever class defined!')

Custom Reciprocal Rank Fusion and Ensemble Retriever class defined!


## 🔧 Step 5 — Define 6 Tools

| # | Tool | Type | Trigger |
|---|------|------|--------|
| 1 | `hybrid_rag_retriever` | Hybrid RAG | PRIMARY — most medical questions |
| 2 | `faiss_rag_retriever` | Semantic RAG | Conceptual / broad questions |
| 3 | `bm25_rag_retriever` | Keyword RAG | Exact term / drug name searches |
| 4 | `wikipedia_live_tool` | Live Search | Rare conditions / extra depth |
| 5 | `symptom_checker` | Rule-based | When symptoms are mentioned |
| 6 | `drug_information` | Rule-based | When a drug name is mentioned |

In [108]:
import wikipediaapi

@tool
def faiss_rag_retriever(query: str) -> str:
    """Semantic/dense vector search in local FAISS medical knowledge base. Best for conceptual medical questions about diseases, treatments, and anatomy."""
    return faiss_retriever.invoke(query)

print('Tool 1: faiss_rag_retriever')

@tool
def bm25_rag_retriever(query: str) -> str:
    """Keyword BM25 search in local medical knowledge base. Best when exact medical terms, drug names, or specific condition names are mentioned."""
    return bm25_retriever.invoke(query)

print('Tool 2: bm25_rag_retriever')

@tool
def hybrid_rag_retriever(query: str) -> str:
    """Hybrid RAG: FAISS semantic + BM25 keyword via Reciprocal Rank Fusion. Use this FIRST as the PRIMARY retriever for all medical questions."""
    return ensemble_retriever.invoke(query)

print('Tool 3: hybrid_rag_retriever')

@tool
def wikipedia_live_tool(query: str) -> str:
    """Search Wikipedia live for medical information not in the local knowledge base. Use as fallback when local RAG is insufficient or for rare/recent conditions."""
    wiki_client = wikipediaapi.Wikipedia(language='en', user_agent='MediBot/2.0')
    try:
        page = wiki_client.page(query)
        if page.exists():
            return f"[Source: {page.fullurl}]\n\n{page.text[:3000]}"
        return f"No Wikipedia page found for '{query}'."
    except Exception as e:
        return f'Wikipedia error: {e}'

print('Tool 4: wikipedia_live_tool')

@tool
def symptom_checker(symptoms: str) -> str:
    """Analyze comma-separated symptoms, return possible conditions with urgency levels. Always use when user mentions symptoms."""
    symptom_map = {
        'fever':               {'conditions': ['Influenza', 'COVID-19', 'Malaria', 'Typhoid', 'Dengue'], 'urgency': 'Medium'},
        'cough':               {'conditions': ['Asthma', 'COVID-19', 'Tuberculosis', 'Bronchitis'], 'urgency': 'Low-Medium'},
        'chest pain':          {'conditions': ['Heart disease', 'Angina', 'Pneumonia', 'GERD'], 'urgency': '🚨 HIGH — Seek immediate care'},
        'headache':            {'conditions': ['Migraine', 'Hypertension', 'Tension headache', 'Meningitis'], 'urgency': 'Low-Medium'},
        'fatigue':             {'conditions': ['Anemia', 'Diabetes', 'Hypothyroidism', 'Depression'], 'urgency': 'Low'},
        'shortness of breath': {'conditions': ['Asthma', 'Heart failure', 'COVID-19', 'Pulmonary embolism'], 'urgency': '🚨 HIGH — Seek immediate care'},
        'frequent urination':  {'conditions': ['Diabetes mellitus', 'UTI', 'Prostate issues'], 'urgency': 'Medium'},
        'weight loss':         {'conditions': ['Diabetes', 'Cancer', 'Tuberculosis', 'Hyperthyroidism'], 'urgency': 'Medium-High'},
        'joint pain':          {'conditions': ['Arthritis', 'Gout', 'Lupus', 'Fibromyalgia'], 'urgency': 'Low-Medium'},
        'rash':                {'conditions': ['Eczema', 'Psoriasis', 'Allergic reaction', 'Lupus'], 'urgency': 'Low'},
        'nausea':              {'conditions': ['Gastritis', 'Food poisoning', 'Pregnancy', 'Migraine'], 'urgency': 'Low-Medium'},
        'dizziness':           {'conditions': ['Hypertension', 'Anemia', 'Inner ear disorder', 'Stroke'], 'urgency': 'Medium'},
        'vomiting':            {'conditions': ['Gastritis', 'Food poisoning', 'Appendicitis', 'Migraine'], 'urgency': 'Medium'},
        'abdominal pain':      {'conditions': ['Gastritis', 'Appendicitis', 'IBS', 'Kidney stones'], 'urgency': 'Medium-High'},
        'back pain':           {'conditions': ['Muscle strain', 'Herniated disc', 'Kidney stones', 'Sciatica'], 'urgency': 'Low-Medium'},
        'swelling':            {'conditions': ['Heart failure', 'Kidney disease', 'DVT', 'Lymphedema'], 'urgency': 'Medium'},
        'blurred vision':      {'conditions': ['Diabetes', 'Hypertension', 'Glaucoma', 'Stroke'], 'urgency': '🚨 HIGH — Seek immediate care'},
        'confusion':           {'conditions': ['Stroke', 'Hypoglycemia', 'Dementia', 'Encephalitis'], 'urgency': '🚨 HIGH — Seek immediate care'},
    }
    entered = [s.strip().lower() for s in symptoms.split(',')]
    result = {}
    for sym in entered:
        for key, data in symptom_map.items():
            if key in sym:
                result[key] = data
    if not result:
        return 'No matches found. Describe symptoms more clearly or consult a healthcare professional.'
    lines = [' Symptom Analysis (Educational Only):\n']
    for sym, data in result.items():
        lines.append(f'  • {sym.capitalize()}')
        lines.append(f'    Possible: {', '.join(data["conditions"])}')
        lines.append(f'    Urgency: {data["urgency"]}')
    lines.append('\n Educational only. Always consult a licensed physician.')
    return '\n'.join(lines)

print('Tool 5: symptom_checker')

@tool
def drug_information(drug_name: str) -> str:
    """Get drug details: class, uses, dosage, side effects, contraindications, interactions. Use when user asks about a specific medication."""
    drug_db = {
        'paracetamol': {'class': 'Analgesic/Antipyretic', 'uses': 'Fever, mild-moderate pain', 'dosage': '500-1000mg every 4-6h, max 4g/day', 'side_effects': 'Rare at normal doses; overdose causes liver damage', 'contraindications': 'Severe liver disease', 'interactions': 'Warfarin (high doses)'},
        'ibuprofen': {'class': 'NSAID', 'uses': 'Pain, fever, inflammation', 'dosage': '200-400mg every 4-6h, max 1200mg/day OTC', 'side_effects': 'GI upset, ulcers, increased BP, kidney issues', 'contraindications': 'Peptic ulcer, kidney disease, pregnancy (3rd trimester)', 'interactions': 'Aspirin, warfarin, ACE inhibitors'},
        'amoxicillin': {'class': 'Penicillin Antibiotic', 'uses': 'Bacterial infections: respiratory, UTI, ear, skin', 'dosage': '250-500mg every 8h or 875mg every 12h', 'side_effects': 'Diarrhea, nausea, rash, yeast overgrowth', 'contraindications': 'Penicillin allergy', 'interactions': 'Warfarin, oral contraceptives'},
        'metformin': {'class': 'Biguanide Antidiabetic', 'uses': 'Type 2 diabetes (first-line)', 'dosage': 'Start 500mg twice daily, max 2550mg/day', 'side_effects': 'GI upset (temporary); rare: lactic acidosis', 'contraindications': 'Kidney failure (eGFR<30), liver disease', 'interactions': 'Alcohol, iodinated contrast, cimetidine'},
        'aspirin': {'class': 'Salicylate NSAID / Antiplatelet', 'uses': 'Pain, fever; low-dose cardiovascular protection', 'dosage': 'Pain: 325-650mg. Cardio: 75-100mg/day', 'side_effects': 'GI irritation, tinnitus (high dose), bleeding', 'contraindications': 'Children <16 (Reye syndrome), peptic ulcer', 'interactions': 'Warfarin, ibuprofen, SSRIs'},
        'insulin': {'class': 'Hormone / Injectable Antidiabetic', 'uses': 'Type 1 DM (essential), Type 2 DM (refractory)', 'dosage': 'Individualized — medical supervision required', 'side_effects': 'Hypoglycemia, weight gain, injection site reactions', 'contraindications': 'Hypoglycemia', 'interactions': 'Beta-blockers, corticosteroids, alcohol'},
    }
    key = drug_name.lower().strip()
    matched = next((k for k in drug_db if k in key or key in k), None)
    if not matched:
        return f"No local data for '{drug_name}'. Try hybrid_rag_retriever or wikipedia_live_tool."
    i = drug_db[matched]
    return (
        f" {matched.capitalize()}\n"
        f"  Class: {i['class']}\n"
        f"  Uses: {i['uses']}\n"
        f"  Dosage: {i['dosage']}\n"
        f"  Side Effects: {i['side_effects']}\n"
        f"  Contraindications: {i['contraindications']}\n"
        f"  Key Interactions: {i['interactions']}\n\n"
        f" Follow your doctor's/pharmacist's instructions."
    )

print(' Tool 6: drug_information')

tools = [hybrid_rag_retriever, faiss_rag_retriever, bm25_rag_retriever, wikipedia_live_tool, symptom_checker, drug_information]
print(f'\n {len(tools)} tools ready!')

Tool 1: faiss_rag_retriever
Tool 2: bm25_rag_retriever
Tool 3: hybrid_rag_retriever
Tool 4: wikipedia_live_tool
Tool 5: symptom_checker
 Tool 6: drug_information

 6 tools ready!


## 🤖 Step 6 — MultiModel LLM Factory

Switch between Gemini, GPT-4o-mini, and Groq Llama-3.3-70b at runtime.

In [109]:
MODEL_CONFIGS = {
    'gemini': {
        'label': ' Gemini 2.5 Flash (Google)',
        'requires': 'GOOGLE_API_KEY',
        'factory': lambda: ChatGoogleGenerativeAI(
            model='gemini-2.5-flash',
            google_api_key=os.environ['GOOGLE_API_KEY'],
            temperature=0.2,
            max_output_tokens=1024,
            convert_system_message_to_human=True
        )
    },
    'openai': {
        'label': 'GPT-4o-mini (OpenAI)',
        'requires': 'OPENAI_API_KEY',
        'factory': lambda: ChatOpenAI(
            model='gpt-4o-mini',
            api_key=os.environ['OPENAI_API_KEY'],
            temperature=0.2,
            max_tokens=1024
        ) if ChatOpenAI else None
    },
    'groq': {
        'label': ' Llama-3.3-70b (Groq)',
        'requires': 'GROQ_API_KEY',
        'factory': lambda: ChatGroq(
            model='llama-3.3-70b-versatile',
            api_key=os.environ['GROQ_API_KEY'],
            temperature=0.2,
            max_tokens=1024
        ) if ChatGroq else None
    },
}

def get_available_models():
    return {n: c for n, c in MODEL_CONFIGS.items() if os.environ.get(c['requires'])}

available = get_available_models()
print('Available Models:')
for name, cfg in available.items():
    print(f'   {cfg["label"]}')

current_model_name = list(available.keys())[0]
print(f'\n Default: {MODEL_CONFIGS[current_model_name]["label"]}')

Available Models:
    Gemini 2.5 Flash (Google)
    Llama-3.3-70b (Groq)

 Default:  Gemini 2.5 Flash (Google)


## 🔗 Step 7 — LangGraph Agent with Tool Attribution

In [110]:
SYSTEM_PROMPT = """
You are MediBot, an expert Medical AI Assistant with a Hybrid RAG system.

## Tools Available:
1. hybrid_rag_retriever — PRIMARY: FAISS semantic + BM25 keyword. Use FIRST for all medical questions.
2. faiss_rag_retriever — Dense semantic search. Use for conceptual questions.
3. bm25_rag_retriever — Keyword search. Use for exact medical terms/drug names.
4. wikipedia_live_tool — Live Wikipedia. Use as fallback for rare/recent conditions.
5. symptom_checker — ALWAYS call when the user mentions any symptoms.
6. drug_information — ALWAYS call when a specific drug/medication is named.

## Rules:
- Start every medical question with hybrid_rag_retriever.
- Call symptom_checker immediately if symptoms are described.
- Call drug_information when a drug is named.
- Use wikipedia_live_tool for extra depth or unknown topics.
- Structure answers with headers and bullet points.
- Always end with a '🔧 Sources & Tools Used' section listing each tool invoked.
- Finish with the medical disclaimer.

*Medical Disclaimer: For educational purposes only. Always consult a qualified healthcare professional.*
"""

class AgentState(TypedDict):
    messages:   Annotated[list, add_messages]
    tools_used: List[str]

def build_agent(model_name: str):
    cfg = MODEL_CONFIGS[model_name]
    llm = cfg['factory']()
    if llm is None:
        raise ValueError(f'Model {model_name} unavailable.')
    llm_with_tools = llm.bind_tools(tools)
    tool_node = ToolNode(tools)

    def call_model(state: AgentState):
        msgs = list(state['messages'])
        if not any(isinstance(m, SystemMessage) for m in msgs):
            msgs = [SystemMessage(content=SYSTEM_PROMPT)] + msgs
        response = llm_with_tools.invoke(msgs)
        return {'messages': [response]}

    def run_tools(state: AgentState):
        last = state['messages'][-1]
        used = list(state.get('tools_used', []))
        if hasattr(last, 'tool_calls') and last.tool_calls:
            for tc in last.tool_calls:
                name = tc.get('name', 'unknown')
                if name not in used:
                    used.append(name)
        result = tool_node.invoke(state)
        result['tools_used'] = used
        return result

    def should_continue(state: AgentState):
        last = state['messages'][-1]
        return 'tools' if (hasattr(last, 'tool_calls') and last.tool_calls) else END

    graph = StateGraph(AgentState)
    graph.add_node('agent', call_model)
    graph.add_node('tools', run_tools)
    graph.set_entry_point('agent')
    graph.add_conditional_edges('agent', should_continue, {'tools': 'tools', END: END})
    graph.add_edge('tools', 'agent')
    return graph.compile()

print(f'Building agent with {MODEL_CONFIGS[current_model_name]["label"]}...')
agent = build_agent(current_model_name)
print('LangGraph MultiModel Medical Agent compiled!')

Building agent with  Gemini 2.5 Flash (Google)...
LangGraph MultiModel Medical Agent compiled!


## 💬 Step 8 — Chat Helper with Tool Attribution

In [111]:
TOOL_LABELS = {
    'hybrid_rag_retriever': 'Hybrid RAG (FAISS + BM25)',
    'faiss_rag_retriever':  'FAISS Semantic RAG',
    'bm25_rag_retriever':   'BM25 Keyword RAG',
    'wikipedia_live_tool':  'Live Wikipedia',
    'symptom_checker':      'Symptom Checker',
    'drug_information':     'Drug Information DB',
}

TOOL_DESCRIPTIONS = {
    'hybrid_rag_retriever': 'FAISS semantic + BM25 keyword via Reciprocal Rank Fusion',
    'faiss_rag_retriever':  'Dense vector similarity — local Wikipedia KB',
    'bm25_rag_retriever':   'TF-IDF keyword matching — local Wikipedia KB',
    'wikipedia_live_tool':  'Real-time Wikipedia API lookup',
    'symptom_checker':      'Rule-based symptom-to-condition mapper with urgency levels',
    'drug_information':     'Local drug DB: dosage, side effects, contraindications, interactions',
}

conversation_history = []

def switch_model(model_name: str):
    global agent, current_model_name
    available = get_available_models()
    if model_name not in available:
        print(f'"{model_name}" not available. Options: {list(available.keys())}')
        return
    print(f'🔄 Switching to {MODEL_CONFIGS[model_name]["label"]}...')
    agent = build_agent(model_name)
    current_model_name = model_name
    print(f' Now using: {MODEL_CONFIGS[model_name]["label"]}')

def ask_medibot(question: str, verbose: bool = True):
    global conversation_history
    conversation_history.append(HumanMessage(content=question))
    try:
        result = agent.invoke(
            {'messages': conversation_history, 'tools_used': []},
            config={'recursion_limit': 25},
        )
    except Exception as e:
        print(f' Agent error: {e}')
        return

    ai_msgs = [m for m in result['messages'] if isinstance(m, AIMessage)]
    if not ai_msgs:
        print(' No AI response received.')
        return

    final_msg = ai_msgs[-1]
    reply = final_msg.content if isinstance(final_msg.content, str) else str(final_msg.content)
    tools_used = result.get('tools_used', [])
    conversation_history.append(final_msg)

    if verbose:
        model_label = MODEL_CONFIGS[current_model_name]['label']
        print(f'\n{"="*80}')
        print(f'🧑 You: {question}')
        print(f'🤖 Model: {model_label}\n')
        print('🏥 MediBot:')
        print(reply)
        print()
        print('─' * 40)
        if tools_used:
            print('🔧 RAG Tools & Sources Used:')
            for t in tools_used:
                print(f'  ✓ {TOOL_LABELS.get(t, t)}')
                print(f'    └─ {TOOL_DESCRIPTIONS.get(t, "")}')
        else:
            print('  (No RAG tools — answered from model knowledge)')
        print(f'{"="*80}\n')

    return reply, tools_used

def reset_conversation():
    global conversation_history
    conversation_history = []
    print('🔄 Conversation cleared.')

print('MediBot ready!')
print("ask_medibot('question') — ask anything")
print("switch_model('gemini'/'openai'/'groq') — change LLM")
print("reset_conversation() — clear history")

MediBot ready!
ask_medibot('question') — ask anything
switch_model('gemini'/'openai'/'groq') — change LLM
reset_conversation() — clear history


## 🧪 Step 9 — Demo Queries (Each Shows Tools Used)

In [112]:
from langchain_core.retrievers import BaseRetriever
from langchain_core.documents import Document
from typing import List
from collections import defaultdict
from pydantic import ConfigDict

# Reciprocal Rank Fusion function (moved from e711cd8a)
def reciprocal_rank_fusion(results: list[list[Document]], k=60):
    """Perform Reciprocal Rank Fusion on a list of lists of documents."""
    fused_scores = defaultdict(float)
    unique_docs = {}
    for r_idx, result_list in enumerate(results):
        for rank, doc in enumerate(result_list):
            doc_id = (doc.page_content, doc.metadata.get('source'))
            if doc_id not in unique_docs:
                unique_docs[doc_id] = doc
            fused_scores[doc_id] += 1 / (k + rank)

    sorted_doc_ids = sorted(fused_scores.keys(), key=lambda x: fused_scores[x], reverse=True)
    fused_docs = [unique_docs[doc_id] for doc_id in sorted_doc_ids]
    return fused_docs


class CustomEnsembleRetriever(BaseRetriever):
    """Custom Ensemble Retriever using Reciprocal Rank Fusion, inheriting from BaseRetriever."""
    retrievers: list # The sub-retrievers (e.g., FAISS, BM25)
    weights: list = None # Weights for future enhancements, RRF inherently handles ranking
    k: int = 60 # Parameter for Reciprocal Rank Fusion

    model_config = ConfigDict(arbitrary_types_allowed=True)

    def _get_relevant_documents(self, query: str) -> List[Document]:
        results = []
        for retriever in self.retrievers:
            # Changed from get_relevant_documents to invoke for broader compatibility
            results.append(retriever.invoke(query))

        fused_documents = reciprocal_rank_fusion(results, k=self.k)
        return fused_documents

    async def _aget_relevant_documents(self, query: str) -> List[Document]:
        # This needs to be implemented if asynchronous calls are expected
        raise NotImplementedError("Asynchronous retrieval not yet implemented for CustomEnsembleRetriever")

print('Updated Custom Reciprocal Rank Fusion and Ensemble Retriever class defined!')

Updated Custom Reciprocal Rank Fusion and Ensemble Retriever class defined!


In [113]:
# This cell is no longer needed as 'ensemble_retriever' is already correctly defined in cell R9feFBKte2L1.
# The 'ensemble_retriever' variable created there is now a fully functional CustomEnsembleRetriever instance.

In [114]:
# Disease question → expects hybrid_rag_retriever
ask_medibot('What is diabetes mellitus and how is it treated?')


🧑 You: What is diabetes mellitus and how is it treated?
🤖 Model:  Gemini 2.5 Flash (Google)

🏥 MediBot:
[{'type': 'text', 'text': "## Understanding Diabetes Mellitus\n\nDiabetes mellitus, commonly known as diabetes, is a collection of endocrine diseases characterized by persistently high blood sugar levels. This condition arises either from the pancreas producing insufficient insulin or the body's cells becoming unresponsive to insulin's effects.\n\n### Types of Diabetes:\n*   **Type 1 Diabetes:** This is an autoimmune condition where the body's immune system attacks and destroys the beta cells in the pancreas, which are responsible for insulin production. It typically manifests from birth or early in life.\n*   **Type 2 Diabetes:** In this more common type, the body develops resistance to insulin, preventing glucose from being absorbed by cells and leading to its accumulation in the bloodstream.\n*   **Gestational Diabetes:** This form can occur during pregnancy and usually resolves 

('[{\'type\': \'text\', \'text\': "## Understanding Diabetes Mellitus\\n\\nDiabetes mellitus, commonly known as diabetes, is a collection of endocrine diseases characterized by persistently high blood sugar levels. This condition arises either from the pancreas producing insufficient insulin or the body\'s cells becoming unresponsive to insulin\'s effects.\\n\\n### Types of Diabetes:\\n*   **Type 1 Diabetes:** This is an autoimmune condition where the body\'s immune system attacks and destroys the beta cells in the pancreas, which are responsible for insulin production. It typically manifests from birth or early in life.\\n*   **Type 2 Diabetes:** In this more common type, the body develops resistance to insulin, preventing glucose from being absorbed by cells and leading to its accumulation in the bloodstream.\\n*   **Gestational Diabetes:** This form can occur during pregnancy and usually resolves after childbirth.\\n\\n### Common Symptoms:\\nClassic symptoms of diabetes include:\\n*

In [115]:
# Symptoms → expects symptom_checker + hybrid_rag
ask_medibot('I have fever, cough, and fatigue. What could it be?')

 Agent error: Error calling model 'gemini-2.5-flash' (RESOURCE_EXHAUSTED): 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. \n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 20, model: gemini-2.5-flash\nPlease retry in 1.80710241s.', 'status': 'RESOURCE_EXHAUSTED', 'details': [{'@type': 'type.googleapis.com/google.rpc.Help', 'links': [{'description': 'Learn more about Gemini API quotas', 'url': 'https://ai.google.dev/gemini-api/docs/rate-limits'}]}, {'@type': 'type.googleapis.com/google.rpc.QuotaFailure', 'violations': [{'quotaMetric': 'generativelanguage.googleapis.com/generate_content_free_tier_requests', 'quotaId': 'GenerateRequestsPerDayPerProjectPerModel-FreeTier', 'quotaDimen

In [116]:
# Drug question → expects drug_information + bm25/hybrid
ask_medibot('What are the side effects and dosage of ibuprofen?')

 Agent error: Error calling model 'gemini-2.5-flash' (RESOURCE_EXHAUSTED): 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. \n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 20, model: gemini-2.5-flash\nPlease retry in 28.239136971s.', 'status': 'RESOURCE_EXHAUSTED', 'details': [{'@type': 'type.googleapis.com/google.rpc.Help', 'links': [{'description': 'Learn more about Gemini API quotas', 'url': 'https://ai.google.dev/gemini-api/docs/rate-limits'}]}, {'@type': 'type.googleapis.com/google.rpc.QuotaFailure', 'violations': [{'quotaMetric': 'generativelanguage.googleapis.com/generate_content_free_tier_requests', 'quotaId': 'GenerateRequestsPerDayPerProjectPerModel-FreeTier', 'quotaDim

In [117]:
# Rare topic → expects wikipedia_live_tool
ask_medibot('Tell me about Kawasaki disease in children.')

 Agent error: Error calling model 'gemini-2.5-flash' (RESOURCE_EXHAUSTED): 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. \n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 20, model: gemini-2.5-flash\nPlease retry in 54.023267955s.', 'status': 'RESOURCE_EXHAUSTED', 'details': [{'@type': 'type.googleapis.com/google.rpc.Help', 'links': [{'description': 'Learn more about Gemini API quotas', 'url': 'https://ai.google.dev/gemini-api/docs/rate-limits'}]}, {'@type': 'type.googleapis.com/google.rpc.QuotaFailure', 'violations': [{'quotaMetric': 'generativelanguage.googleapis.com/generate_content_free_tier_requests', 'quotaId': 'GenerateRequestsPerDayPerProjectPerModel-FreeTier', 'quotaDim

## 🔄 Step 10 — Model Comparison (Same Question, 3 LLMs)

In [118]:
question = 'What are the main symptoms and risk factors of hypertension?'
available = get_available_models()
print(f'Comparing {len(available)} model(s) on: "{question}"\n')

for model_name in available:
    reset_conversation()
    switch_model(model_name)
    ask_medibot(question)

Comparing 2 model(s) on: "What are the main symptoms and risk factors of hypertension?"

🔄 Conversation cleared.
🔄 Switching to  Gemini 2.5 Flash (Google)...
 Now using:  Gemini 2.5 Flash (Google)
 Agent error: Error calling model 'gemini-2.5-flash' (RESOURCE_EXHAUSTED): 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. \n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 20, model: gemini-2.5-flash\nPlease retry in 20.793037984s.', 'status': 'RESOURCE_EXHAUSTED', 'details': [{'@type': 'type.googleapis.com/google.rpc.Help', 'links': [{'description': 'Learn more about Gemini API quotas', 'url': 'https://ai.google.dev/gemini-api/docs/rate-limits'}]}, {'@type': 'type.googleapis.com/google

## 🖥️ Step 11 — Interactive Chat Widget

In [119]:
import ipywidgets as widgets
from IPython.display import clear_output

reset_conversation()
available_models_dict = get_available_models()
model_options = {cfg['label']: name for name, cfg in available_models_dict.items()}

header = widgets.HTML(
    '<h2 style="color:#1a73e8;font-family:monospace;">🏥 MediBot v2.0 — MultiModel Medical AI</h2>'
    '<p style="color:#555;font-size:12px;margin:0">'
    'Hybrid RAG (FAISS + BM25) · 6 Tools · Multi-LLM · Tool Attribution</p>'
)

model_dropdown = widgets.Dropdown(
    options=list(model_options.keys()),
    value=list(model_options.keys())[0],
    description='🤖 LLM:',
    layout=widgets.Layout(width='38%')
)

txt_input = widgets.Text(
    placeholder='Ask a medical question...',
    layout=widgets.Layout(width='52%')
)

btn_ask   = widgets.Button(description='Ask', button_style='primary', layout=widgets.Layout(width='10%'))
btn_clear = widgets.Button(description='Clear', button_style='warning', layout=widgets.Layout(width='10%'))

status_lbl = widgets.HTML(value='')

out = widgets.Output(layout=widgets.Layout(
    border='1px solid #e0e0e0',
    padding='12px',
    min_height='250px',
    max_height='700px',
    overflow_y='auto'
))

def on_model_change(change):
    switch_model(model_options[change['new']])

def on_ask(b):
    q = txt_input.value.strip()
    if not q: return
    txt_input.value = ''
    btn_ask.disabled = True
    status_lbl.value = '<span style="color:#1a73e8;">⏳ Thinking...</span>'
    with out:
        ask_medibot(q)
    btn_ask.disabled = False
    status_lbl.value = ''

def on_clear(b):
    reset_conversation()
    with out:
        clear_output()
        print('Chat cleared.')
    status_lbl.value = ''

model_dropdown.observe(on_model_change, names='value')
btn_ask.on_click(on_ask)
btn_clear.on_click(on_clear)
txt_input.on_submit(on_ask)

display(
    header,
    model_dropdown,
    widgets.HBox([txt_input, btn_ask, btn_clear]),
    status_lbl,
    out
)

with out:
    print('👋 Welcome to MediBot v2.0 — MultiModel Edition!')
    print()
    print('  New features:')
    print('  Hybrid RAG: FAISS (semantic) + BM25 (keyword)')
    print('  Switch LLM from the dropdown above')
    print('  Every answer shows which RAG tools were used')
    print()
    print('  Try:')
    print('  • What is diabetes and how is it treated?')
    print('  • I have fever, cough, and shortness of breath')
    print('  • What are the side effects of metformin?')
    print('  • Tell me about the immune system')

🔄 Conversation cleared.


HTML(value='<h2 style="color:#1a73e8;font-family:monospace;">🏥 MediBot v2.0 — MultiModel Medical AI</h2><p sty…

Dropdown(description='🤖 LLM:', layout=Layout(width='38%'), options=(' Gemini 2.5 Flash (Google)', ' Llama-3.3-…

HTML(value='')

Output(layout=Layout(border='1px solid #e0e0e0', max_height='700px', min_height='250px', overflow_y='auto', pa…